<a href="https://colab.research.google.com/github/AdityaJare/ML-PRojects/blob/main/HR_Employee_Attrition_ann_keras_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install tensorflow keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 9.3 MB/s eta 0:00:00


In [97]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.callbacks import Callback
from tensorflow.keras import layers
from keras.layers import Dense, Dropout
import keras_tuner as kt
import matplotlib.pyplot as plt


In [70]:
df = pd.read_csv("/WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [71]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


In [72]:
df.shape

(1470, 35)

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [74]:
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"])


In [75]:
df.shape

(1470, 31)

In [76]:
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No" : 0})

In [77]:
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,0,Travel_Frequently,884,Research & Development,23,2,Medical,3,Male,41,4,2,Laboratory Technician,4,Married,2571,12290,4,No,17,3,3,1,17,3,3,5,2,0,3
1466,39,0,Travel_Rarely,613,Research & Development,6,1,Medical,4,Male,42,2,3,Healthcare Representative,1,Married,9991,21457,4,No,15,3,1,1,9,5,3,7,7,1,7
1467,27,0,Travel_Rarely,155,Research & Development,4,3,Life Sciences,2,Male,87,4,2,Manufacturing Director,2,Married,6142,5174,1,Yes,20,4,2,1,6,0,3,6,2,0,3
1468,49,0,Travel_Frequently,1023,Sales,2,3,Medical,4,Male,63,2,2,Sales Executive,2,Married,5390,13243,2,No,14,3,4,0,17,3,2,9,6,0,8


In [78]:
X = df.drop(columns="Attrition", axis=1)
y = df["Attrition"]

In [79]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= 42, stratify=y)

In [80]:
x_train.head()

,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
1194,47,Travel_Rarely,1225,Sales,2,4,Life Sciences,2,Female,47,4,4,Manager,2,Divorced,15972,21086,6,No,14,3,3,3,29,2,3,3,2,1,2
128,22,Travel_Rarely,594,Research & Development,2,1,Technical Degree,3,Male,100,3,1,Laboratory Technician,4,Married,2523,19299,0,No,14,3,3,1,3,2,3,2,1,2,1
810,46,Travel_Rarely,406,Sales,3,1,Marketing,1,Male,52,3,4,Manager,3,Married,17465,15596,3,No,12,3,4,1,23,3,3,12,9,4,9
478,25,Travel_Rarely,622,Sales,13,1,Medical,2,Male,40,3,1,Sales Representative,3,Married,2096,26376,1,No,11,3,3,0,7,1,3,7,4,0,6
491,43,Travel_Frequently,1001,Research & Development,9,5,Medical,4,Male,72,3,2,Laboratory Technician,3,Divorced,5679,19627,3,Yes,13,3,2,1,10,3,3,8,7,4,7


In [81]:
numeric_cols= x_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
onehot_cols = ['Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
ordinal_cols = ['BusinessTravel']

In [82]:
numeric_cols = [c for c in numeric_cols if c not in onehot_cols + ordinal_cols]

In [83]:
# individual preprocessing pipeline

scaling_pipe = Pipeline([
    ("Scaling", StandardScaler())
])

onehot_pipe = Pipeline([
    ("OneHot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

ordinal_pipe = Pipeline([
    ("Ordinal", OrdinalEncoder(categories=[['Non-Travel', 'Travel_Rarely', 'Travel_Frequently']]))
])

In [84]:
# building full preprocessing pipeline

preprocessing = ColumnTransformer([
    ("Scaling", scaling_pipe, numeric_cols),
    ("onehot", onehot_pipe, onehot_cols),
    ("ordinal", ordinal_pipe, ordinal_cols)
])

In [85]:
x_train_preprocessed = preprocessing.fit_transform(x_train)
x_test_preprocessed = preprocessing.transform(x_test)

In [86]:
x_train_preprocessed.shape

(1176, 43)

In [87]:
n_features = x_train_preprocessed.shape[1]

In [92]:
def build_model(hp):

  model = keras.Sequential()

  model.add(layers.Input(shape=(n_features,)))

  for i in range(hp.Int("num_layers" , 1, 3)):

    model.add(Dense(
        units=hp.Int(f"unit{i} ", min_value=16, max_value=128, step=16),
        activation="relu"
    ))

    model.add(Dropout(hp.Float(f"Dropout {i} : ", min_value=0.1, max_value=0.5, step=0.1)))

  model.add(Dense(1, activation="sigmoid"))

  hp_learning_rate= hp.Choice("learning_rate" , values=[0.1, 0.01, 0.001])
  hp_optimizer =  hp.Choice("optimizer" , values=["adam", "rmsprop", "nadam", "sgd"])

  if hp_optimizer == "adam":
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
  elif hp_optimizer == "rmsprop":
    optimizer = keras.optimizers.RMSprop(learning_rate=hp_learning_rate)
  elif hp_optimizer == "nadam":
    optimizer = keras.optimizers.Nadam(learning_rate=hp_learning_rate)
  elif hp_optimizer == "sgd":
    optimizer = keras.optimizers.SGD(learning_rate=hp_learning_rate)

  model.compile(
    optimizer = optimizer,
    loss="binary_crossentropy",  # standard loss for binary classification
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
  )
  return model


In [98]:
tuner = kt.RandomSearch(
    build_model,
    objective = kt.Objective("val_auc", direction="max"),
    max_trials = 10,
    executions_per_trial=1,
    directory="keras_tuner_dir",
    project_name="attrition_ann"

)

In [108]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_auc", patience=5, mode="max", restore_best_weights=True
)

In [103]:
tuner.search(
    x_train_preprocessed, y_train,
    validation_split=0.2,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

Trial 10 Complete [00h 00m 06s]
val_auc: 0.8466198444366455

Best val_auc So Far: 0.8744897246360779
Total elapsed time: 00h 01m 43s


In [104]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best hyperparameters found:")
print(best_hp.values)

Best hyperparameters found:
{'num_layers': 1, 'unit0 ': 128, 'Dropout 0 : ': 0.5, 'learning_rate': 0.01, 'optimizer': 'nadam', 'unit1 ': 80, 'Dropout 1 : ': 0.4, 'unit2 ': 64, 'Dropout 2 : ': 0.30000000000000004}


In [109]:
best_model=tuner.hypermodel.build(best_hp)

In [110]:
history = best_model.fit(
    x_train_preprocessed, y_train,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.8330 - auc: 0.6537 - loss: 0.4482 - val_accuracy: 0.8686 - val_auc: 0.8287 - val_loss: 0.3387
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8638 - auc: 0.8133 - loss: 0.3436 - val_accuracy: 0.8686 - val_auc: 0.8455 - val_loss: 0.3227
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8723 - auc: 0.8337 - loss: 0.3250 - val_accuracy: 0.8559 - val_auc: 0.8452 - val_loss: 0.3286
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8872 - auc: 0.8502 - loss: 0.3066 - val_accuracy: 0.8898 - val_auc: 0.8545 - val_loss: 0.3125
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8840 - auc: 0.8772 - loss: 0.2859 - val_accuracy: 0.8941 - val_auc: 0.8401 - val_loss: 0.3144
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8915 - auc: 0.8857 - loss: 0.2770 - val_accuracy: 0.8898 - val_auc: 0.8480 - val_loss: 0.3204
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms

In [112]:
from sklearn.metrics import classification_report

y_pred_proba = best_model.predict(x_test_preprocessed)
y_pred = (y_pred_proba > 0.5).astype(int)

print(classification_report(y_test, y_pred))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
              precision    recall  f1-score   support

           0       0.88      0.97      0.92       247
           1       0.65      0.32      0.43        47

    accuracy                           0.86       294
   macro avg       0.77      0.64      0.68       294
weighted avg       0.85      0.86      0.84       294

